# 💻 ParagAI: Lightweight Local CPU Training & Inference Guide

No GPU? No problem! **ParagAI** is engineered from first principles to be lightweight, modular, and fully functional on standard CPU hardware (Intel Core i3/i5/i7/i9, AMD Ryzen, or Apple Silicon).

> **💡 How to run under `.venv` in VS Code / IDE:**
> 1. Look at the **top right corner** of this notebook editor (where it displays the Python kernel name).
> 2. Click on it and choose **Select Another Kernel...** > **Python Environments...**.
> 3. Select **`.venv`** (`c:\Users\hedpa\Documents\AI\ParagAI\.venv\Scripts\python.exe`).
> *(Note: Even if you keep global Python selected, the code cells below automatically route all execution through `.venv`!)*

---

### 🎯 Key Highlights for CPU Users:
- 🧠 **Minimal RAM Footprint:** Runs comfortably on 8GB–16GB system RAM.
- ⚡ **Multi-Core Threading:** Leverages all CPU cores via PyTorch MKL / OpenMP optimizations.
- 🔄 **Incremental Training:** Train 10–20 minutes daily without slowing down your computer.
- 🚀 **Fast CPU Inference in Ollama:** Ollama uses AVX2 SIMD instructions to generate text at 20–50+ tokens/sec on CPU!
- 🛡️ **Zero Data Loss:** Learn how to add new data day-by-day while keeping past knowledge intact.

## ⚙️ 1. Environment Setup & Auto-Binding to `.venv`
Switches working directory to the project root and automatically binds execution to the `.venv` Python virtual environment.

In [ ]:
import os
import sys
import platform
import torch

# 1. Automatically change working directory to ParagAI repository root if started inside notebooks/
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# 2. Add src to Python path
sys.path.insert(0, os.path.abspath("src"))

# 3. Locate and bind to the workspace virtual environment (.venv)
venv_py = os.path.abspath(os.path.join(".venv", "Scripts", "python.exe"))
if not os.path.exists(venv_py):
    venv_py = os.path.abspath(os.path.join(".venv", "bin", "python"))
py_exec = venv_py if os.path.exists(venv_py) else sys.executable

# 4. Multi-core thread allocation for CPU
num_cores = os.cpu_count() or 4
torch.set_num_threads(num_cores)

print("=" * 60)
print(f"Project Root:      {os.getcwd()}")
print(f"Using venv Python: {py_exec}")
print(f"CPU Logical Cores: {num_cores}")
print(f"PyTorch Threads:   {torch.get_num_threads()}")
print(f"PyTorch Version:   {torch.__version__}")
print("=" * 60)

## 🩺 2. Run Pre-Flight Diagnostics (`doctor.py`)
Verifies your environment, model tensor shapes, and tokenizer roundtrips under `.venv`.

In [ ]:
!{py_exec} scripts/doctor.py

## 📚 3. Prepare Multi-Task SFT Data with Replay Buffer
Prepares your text corpus (stories, conversations, translations, and tools) with replay mixing.

In [ ]:
# Merge all raw text datasets into a single training file under .venv
!{py_exec} scripts/prepare_data.py --no_tags

## 🔤 4. Verify or Train the Indic + English Tokenizer
Uses the pre-trained `checkpoints/tokenizer.json` or trains a new multilingual BPE tokenizer.

In [ ]:
!{py_exec} scripts/train_multilingual_bpe.py \
    --input "data/raw/**/*.txt" \
    --vocab_size 4096 \
    --output checkpoints/tokenizer_indic.json \
    --sample_mb 5

## 🚀 5. Lightweight CPU Training
Runs an efficient CPU training session with small batch sizes and live telemetry progress.

In [ ]:
!{py_exec} scripts/train.py \
    --data data/processed/combined_corpus.txt \
    --config configs/train_config.yaml \
    --model_config configs/model_small.yaml \
    --device cpu \
    --batch_size 16 \
    --max_iters 500

## 💬 6. Interactive Terminal Chat on CPU
Chat with the model directly in Python using standard greedy / nucleus sampling.

In [ ]:
import json
import os
import torch
from paragai.config import ModelConfig
from paragai.inference.generate import Generator
from paragai.model.transformer import ParagAIModel
from paragai.tokenizer.bpe_tokenizer import BPETokenizer
from paragai.tokenizer.char_tokenizer import CharTokenizer
from paragai.tokenizer.indic_bpe import IndicBPETokenizer

device = torch.device('cpu')
checkpoint = torch.load('checkpoints/best_model.pt', map_location=device)
model_config = ModelConfig(**checkpoint['model_config'])
model = ParagAIModel(model_config)
state_dict = {k: v for k, v in checkpoint['model_state_dict'].items() if not k.endswith('.attn.mask')}
model.load_state_dict(state_dict, strict=False)
model.eval()

tok_path = 'checkpoints/tokenizer.json' if os.path.exists('checkpoints/tokenizer.json') else 'checkpoints/tokenizer_indic.json'
with open(tok_path, 'r', encoding='utf-8') as f:
    tok_data = json.load(f)
if tok_data.get('tokenizer_class') == 'indic_bpe':
    tokenizer = IndicBPETokenizer.load(tok_path)
elif tok_data.get('type') == 'bpe':
    tokenizer = BPETokenizer.load(tok_path)
else:
    tokenizer = CharTokenizer.load(tok_path)

generator = Generator(model, tokenizer, device=device)

prompt = "<|im_start|>user\nHello! Tell me about ParagAI.<|im_end|>\n<|im_start|>assistant\n"
response = generator.generate(prompt, max_new_tokens=80, temperature=0.3, include_prompt=False)
print(response.split('<|im_end|>')[0].strip())

## 📊 7. Run Quantitative Benchmark Evaluation (`eval.py`)

In [ ]:
!{py_exec} scripts/eval.py \
    --checkpoint checkpoints/best_model.pt \
    --tokenizer checkpoints/tokenizer.json \
    --tasks multiling,tools,story

## ⚡ 8. Export to GGUF and Run in Ollama (High-Speed CPU Serving)
Even on CPU, Ollama runs GGUF models at lightning speed using llama.cpp's AVX2 optimized CPU kernels.

In [ ]:
# 8.1 Export to SafeTensors
!{py_exec} scripts/export_hf.py --checkpoint checkpoints/best_model.pt --output_dir checkpoints/export_hf --verify

# 8.2 Convert to GGUF with GPT-2 Unicode byte mapping
!{py_exec} scripts/export_gguf.py --input_dir checkpoints/export_hf --tokenizer checkpoints/tokenizer.json --output checkpoints/paragai.gguf

# 8.3 Register into Ollama (Ensure Ollama desktop application is running)
!ollama create paragai -f deploy/Modelfile

# 8.4 Run instantly in terminal or query via CLI
!ollama run paragai:latest "Hello! Who are you?"